In [ ]:
# === Flags de controle de uso dos classificadores ===
USE_REGEX = True
USE_DEEPSEEK = True
USE_GPT35 = True
USE_GEMINI = True
USE_LLAMA = True


In [ ]:
# === Importações dos classificadores ===
from modules.classifiers.regex_classifier import RegexLegalClassifier
from modules.classifiers.deepseek_classifier import DeepSeekLegalClassifier
from modules.classifiers.gpt35_classifier import GPT35LegalClassifier
from modules.classifiers.gemini_classifier import GeminiLegalClassifier
from modules.classifiers.llama_classifier import LlamaLocalClassifier
from modules.classifiers.llama_classifier import ClassificationResult  # usado no fallback do botão


import os
from dotenv import load_dotenv

# Caminho padrão para o .env
dotenv_path = ".\.env"
if os.path.exists(dotenv_path):
    load_dotenv(dotenv_path)
else:
    print("❌ Arquivo .env não encontrado!")


In [ ]:
# === Inicializar LLaMA com supressão de log de carregamento ===
import os, sys, contextlib

llama_model_path = "llama.cpp/models/llama-2-7b/llama-2-7b.Q4_K_M.gguf"

with open(os.devnull, 'w') as fnull, contextlib.redirect_stdout(fnull):
    llama_classifier = LlamaLocalClassifier(model_path=llama_model_path)


In [ ]:
import ipywidgets as widgets
from IPython.display import display, clear_output

def criar_botoes_classificadores(entrada_texto):
    # Criação de botões de classificação
    regex_button = widgets.Button(description="Regex", button_style='info')
    deepseek_button = widgets.Button(description="DeepSeek", button_style='primary')
    gpt_button = widgets.Button(description="GPT-3.5", button_style='warning')
    gemini_button = widgets.Button(description="Gemini", button_style='success')
    llama_button = widgets.Button(description="LLaMA-2", button_style='info')

    # Saída compartilhada
    classificacao_output = widgets.Output()

    # Manipulador
    def on_button_click(classificador_nome):
        def handler(b):
            with classificacao_output:
                clear_output()
                if not entrada_texto.value.strip():
                    print("⚠️ Texto vazio.")
                    return
                if classificador_nome == "llama":
                    resultado = llama_classifier.classify_text(entrada_texto.value)
                else:
                    resultado = ClassificationResult("não previsto", f"⚠️ Classificador {classificador_nome} não implementado", "fallback_local")
                print(f"🔍 Classificação ({classificador_nome}):", resultado.classification)
                print("📡 Status:", resultado.status)
        return handler

    # Eventos
    regex_button.on_click(on_button_click("regex"))
    deepseek_button.on_click(on_button_click("deepseek"))
    gpt_button.on_click(on_button_click("gpt"))
    gemini_button.on_click(on_button_click("gemini"))
    llama_button.on_click(on_button_click("llama"))

    # Exibição
    display(widgets.HBox([regex_button, deepseek_button, gpt_button, gemini_button, llama_button]))
    display(classificacao_output)


In [ ]:
# ✅ Importa o classificador LLaMA local (modelo já deve estar disponível no caminho informado)
import sys
sys.path.append(".")

from modules.classifiers.llama_classifier import LlamaLocalClassifier

# ✅ Caminho correto para o modelo quantizado (.gguf)
llama_model_path = "llama.cpp/models/llama-2-7b/llama-2-7b.Q4_K_M.gguf"

# ✅ Inicializa o classificador LLaMA (uma única vez)
llama_classifier = LlamaLocalClassifier(model_path=llama_model_path)


In [ ]:
import ipywidgets as widgets
from IPython.display import display, clear_output

# Criação de botões de classificação
regex_button = widgets.Button(description="Regex", button_style='info')
deepseek_button = widgets.Button(description="DeepSeek", button_style='primary')
gpt_button = widgets.Button(description="GPT-3.5", button_style='warning')
gemini_button = widgets.Button(description="Gemini", button_style='success')
llama_button = widgets.Button(description="LLaMA-2", button_style='info')

# Saída compartilhada
classificacao_output = widgets.Output()

# Manipulador de eventos para todos os botões
def on_button_click(classificador_nome):
    def handler(b):
        with classificacao_output:
            clear_output()
            if not entrada_texto.value.strip():
                print("⚠️ Texto vazio.")
                return
            if classificador_nome == "llama":
                resultado = llama_classifier.classify_text(entrada_texto.value)
            else:
                # Placeholder - ajuste conforme necessário para outros classificadores
                resultado = ClassificationResult("não previsto", "⚠️ Classificador não implementado", "fallback_local")
            print(f"🔍 Classificação ({classificador_nome}):", resultado.classification)
            print("📡 Status:", resultado.status)
    return handler

# Associar eventos aos botões
regex_button.on_click(on_button_click("regex"))
deepseek_button.on_click(on_button_click("deepseek"))
gpt_button.on_click(on_button_click("gpt"))
gemini_button.on_click(on_button_click("gemini"))
llama_button.on_click(on_button_click("llama"))

# Exibir em linha
display(widgets.HBox([regex_button, deepseek_button, gpt_button, gemini_button, llama_button]))
display(classificacao_output)


In [ ]:
# === Instanciando classificadores disponíveis ===

# Classificador por regex (sempre disponível)
regex_classifier = RegexLegalClassifier() if USE_REGEX else None

# DeepSeek
deepseek_classifier = None
if USE_DEEPSEEK:
    deepseek_key = os.getenv("DEEPSEEK_API_KEY")
    if deepseek_key:
        deepseek_classifier = DeepSeekLegalClassifier(deepseek_key)
    else:
        print("⚠️ DEEPSEEK_API_KEY não definida no .env")

# GPT-3.5 da OpenAI
gpt35_classifier = None
if USE_GPT35:
    gpt_key = os.getenv("OPENAI_API_KEY")
    if gpt_key:
        gpt35_classifier = GPT35LegalClassifier(gpt_key)
    else:
        print("⚠️ OPENAI_API_KEY não definida no .env")

# Gemini da Google
gemini_classifier = None
if USE_GEMINI:
    gemini_key = os.getenv("GOOGLE_API_KEY")
    if gemini_key:
        gemini_classifier = GeminiLegalClassifier(gemini_key)
    else:
        print("⚠️ GOOGLE_API_KEY não definida no .env")


In [ ]:
# Seção de entrada de texto com título
titulo_entrada = widgets.HTML("<h3>📨 Texto do e-mail a ser analisado</h3>")

texto_input = widgets.Textarea(
    value='',
    placeholder='Cole aqui o texto do e-mail para análise...',
    description='Texto:',
    layout=widgets.Layout(width='100%', height='150px')
)

In [ ]:
# === Definição dos Botões de Classificação e Área de Saída ===

import ipywidgets as widgets

# Título da seção de botões
titulo_botoes = widgets.HTML("<h3>⚖️ Classificadores de Texto Jurídico</h3>")

# Botão - Regex
botao_processar_regex = widgets.Button(
    description="🔎 Regex",
    button_style='info',
    layout=widgets.Layout(width='auto'),
    tooltip="Classificar usando expressões regulares"
)

# Botão - DeepSeek
botao_processar_deepseek = widgets.Button(
    description="🧠 DeepSeek",
    button_style='primary',
    layout=widgets.Layout(width='auto'),
    tooltip="Classificar usando o modelo DeepSeek"
)

# Botão - GPT-3.5
botao_processar_gpt35 = widgets.Button(
    description="🤖 GPT-3.5",
    button_style='warning',
    layout=widgets.Layout(width='auto'),
    tooltip="Classificar usando o modelo da OpenAI"
)

# Botão - Gemini
botao_processar_gemini = widgets.Button(
    description="🌟 Gemini",
    button_style='info',   # <-- Usar 'info', 'success', etc.
    layout=widgets.Layout(width='auto'),
    tooltip="Classificar usando o modelo Gemini"
)

# Área de saída dos resultados
saida_classificacao = widgets.Output()

# Agrupamento dos botões em um único container
grupo_botoes_classificadores = widgets.VBox([
    titulo_botoes,
    widgets.HBox([
        botao_processar_regex,
        botao_processar_deepseek,
        botao_processar_gpt35,
        botao_processar_gemini
    ]),
    widgets.HTML("<br>"),
    saida_classificacao
])


In [ ]:
# === Seção de Busca de Termos no Texto ===

titulo_busca = widgets.HTML("<h3>🔍 Busca de Palavras ou Expressões no Texto</h3>")

busca_input = widgets.Text(
    value='',
    placeholder='Digite termos para buscar (separados por ";")',
    description='Buscar:',
    layout=widgets.Layout(width='80%')
)

botao_buscar = widgets.Button(
    description='🔎 Buscar',
    button_style='success',
    layout=widgets.Layout(width='20%')
)

saida_busca = widgets.Output()

def ao_clicar_buscar(b):
    with saida_busca:
        clear_output()
        texto = texto_input.value.strip()
        termos = busca_input.value.strip()
        
        if not texto:
            print("⚠️ Por favor, insira um texto para análise.")
            return
            
        if not termos:
            print("⚠️ Por favor, digite termos para buscar.")
            return

        # Aplica destaque e exibe HTML
        highlighted = highlight_text(texto, termos)
        display(widgets.HTML(f'''
            <div style="border:1px solid #ccc; padding:10px; background:#f9f9f9; margin-top:10px; white-space:pre-wrap;">
                {highlighted}
            </div>
        '''))

        # Conta ocorrências
        counts = count_occurrences(texto, termos)
        print("\n📌 Ocorrências encontradas:")
        for term, count in counts.items():
            print(f" - '{term}': {count} ocorrência(s)")

botao_buscar.on_click(ao_clicar_buscar)


In [ ]:
# === Seção de Avaliação da Classificação ===

titulo_avaliacao = widgets.HTML("<h3>📝 Avaliação da Classificação</h3>")

avaliador_concorda = widgets.RadioButtons(
    options=['Sim', 'Não'],
    description='Concorda?',
    layout=widgets.Layout(width='300px')
)

campo_observacao = widgets.Textarea(
    value='',
    placeholder='Digite uma observação opcional...',
    description='Observação:',
    layout=widgets.Layout(width='100%', height='100px')
)

botao_registrar = widgets.Button(
    description='Registrar Avaliação',
    button_style='info',
    icon='check'
)

saida_avaliacao = widgets.Output()

# Agrupamento para exibição futura
painel_avaliacao = widgets.VBox([
    titulo_avaliacao,
    avaliador_concorda,
    campo_observacao,
    botao_registrar,
    widgets.HTML("<br>"),
    saida_avaliacao
])


In [ ]:
def classificar_texto(texto, method='deepseek'):
    texto = texto.strip()

    if method == 'deepseek' and USE_DEEPSEEK and deepseek_classifier:
        resultado = deepseek_classifier.classify_text(texto)
        return resultado.classification if resultado.status == "sucesso" else "erro"

    elif method == 'regex' and USE_REGEX and regex_classifier:
        resultado = regex_classifier.classify(texto)
        return resultado.classification if resultado.classification else "fora_de_contexto"

    elif method == 'gpt35' and USE_GPT35 and gpt35_classifier:
        resultado = gpt35_classifier.classify_text(texto)
        return resultado.classification if resultado.status == "sucesso" else "erro"
    
    elif method == 'gemini' and USE_GPT35 and gemini_classifier:
        resultado = gemini_classifier.classify_text(texto)
        return resultado.classification if resultado.status == "sucesso" else "erro"    

    return "método não implementado"


In [ ]:
# === Funções de processamento e registro ===

def exibir_resultado(texto, classificacao, method):
    with saida_classificacao:
        clear_output()
        metadados = extrair_metadados(texto)

        print(f"🔍 Classificação ({method}): {classificacao}")
        print(f"📄 Número do Processo: {metadados['Número do Processo']}")

        if metadados['Prazos'] != "Nenhum prazo identificado":
            print("\n⏳ Prazos encontrados:")
            for prazo in metadados['Prazos']:
                print(f"  - {prazo['texto']} ({prazo['dias']} {prazo['tipo']})")

        if metadados['Partes Réus'] != "Nenhuma parte ré identificada":
            print("\n⚖️ Partes Réus identificadas:")
            for i, reu in enumerate(metadados['Partes Réus'], 1):
                tipo = "(réu por ser a última parte listada)" if reu['tipo_indicador'] == "ultima_parte" else ""
                print(f"  {i}. {reu['nome']} {tipo}")

def ao_clicar_registrar(b):
    with saida_avaliacao:
        clear_output()
        texto = texto_input.value.strip()
        classificacao = classificar_texto(texto)
        metadados = extrair_metadados(texto)
        concorda = avaliador_concorda.value
        observacao = campo_observacao.value
        registrar_avaliacao(texto, classificacao, metadados, concorda, observacao)
        print("✅ Avaliação registrada com sucesso!")

# === Mapeamento de botões para métodos ===

def processar_com_metodo(method_id):
    texto = texto_input.value.strip()
    if not texto:
        with saida_classificacao:
            clear_output()
            print("⚠️ Por favor, insira um texto para análise.")
        return

    classificacao = classificar_texto(texto, method=method_id)
    exibir_resultado(texto, classificacao, method_id)

# Mapeamento dos botões para os métodos corretos
method_handlers = {
    botao_processar_regex: "regex",
    botao_processar_deepseek: "deepseek",
    botao_processar_gpt35: "gpt35",
    botao_processar_gemini: "gemini"   # Agora adicionamos o Gemini corretamente!
}

for botao, metodo in method_handlers.items():
    botao.on_click(lambda b, m=metodo: processar_com_metodo(m))



# Conclusão

- Esta etapa valida o comportamento do modelo e da interface.
- As classificações são registradas com avaliação.
- Agora é possível comparar os resultados entre Regex e DeepSeek.


In [ ]:

# === Seção: Teste de Expressão Regular (atualizada com .search) ===

import re
regex_input = widgets.Text(
    value='',
    placeholder='Digite a expressão regex',
    description='Regex:',
    layout=widgets.Layout(width='80%')
)

texto_teste_input = widgets.Textarea(
    value='',
    placeholder='Digite ou cole o texto para testar',
    description='Texto:',
    layout=widgets.Layout(width='100%', height='100px')
)

saida_regex = widgets.Output()

def testar_regex(b):
    with saida_regex:
        clear_output()
        try:
            pattern = re.compile(regex_input.value)
            match = pattern.search(texto_teste_input.value)
            if match:
                print("🟢 Condições satisfeitas. Regex CASOU com o texto.")
            else:
                print("⚠️ Nenhuma correspondência encontrada.")
        except Exception as e:
            print(f"Erro na regex: {e}")

botao_testar_regex = widgets.Button(
    description="Testar Regex",
    button_style='info'
)
botao_testar_regex.on_click(testar_regex)

In [ ]:
# === Montagem final das abas ===

# Aba 1: Classificação
aba_classificacao = widgets.VBox([
    widgets.HTML("<h2>📨 Classificação de Texto Jurídico</h2>"),
    texto_input,
    widgets.HTML("<br>"),
    widgets.HTML("<h4>⚖️ Escolha o classificador:</h4>"),
    widgets.HBox([
        botao_processar_regex, 
        botao_processar_deepseek, 
        botao_processar_gpt35,
        botao_processar_gemini   # <-- Agora incluímos o botão Gemini corretamente!
    ]),
    widgets.HTML("<br>"),
    saida_classificacao
])

# Aba 2: Busca
aba_busca = widgets.VBox([
    widgets.HTML("<h2>🔍 Busca de Termos</h2>"),
    widgets.HBox([busca_input, botao_buscar]),
    widgets.HTML("<br>"),
    saida_busca
])

# Aba 3: Regex
aba_regex = widgets.VBox([
    widgets.HTML("<h2>🧪 Teste de Expressão Regular</h2>"),
    regex_input,
    texto_teste_input,
    botao_testar_regex,
    widgets.HTML("<br>"),
    saida_regex
])

# Aba 4: Avaliação
aba_avaliacao = widgets.VBox([
    widgets.HTML("<h2>📝 Avaliação da Classificação</h2>"),
    avaliador_concorda,
    campo_observacao,
    botao_registrar,
    widgets.HTML("<br>"),
    saida_avaliacao
])

# Criação do painel com abas
abas = widgets.Tab(children=[
    aba_classificacao,
    aba_busca,
    aba_regex,
    aba_avaliacao
])

abas.set_title(0, "Classificação")
abas.set_title(1, "Busca")
abas.set_title(2, "Regex")
abas.set_title(3, "Avaliação")

display(abas)


In [1]:
# === Seção: Edição Dinâmica do Prompt com Recarregamento dos Classificadores ===

import os
import ipywidgets as widgets
from IPython.display import display, clear_output
from modules.prompts.prompt_definitions import PROMPT_CUSTOM_FILE

# Funções de manipulação de prompt
def carregar_prompt_padrao() -> str:
    from modules.prompts.prompt_definitions import CLASSIFICATION_PROMPT
    return CLASSIFICATION_PROMPT

def carregar_prompt_custom() -> str:
    if os.path.exists(PROMPT_CUSTOM_FILE):
        with open(PROMPT_CUSTOM_FILE, "r", encoding="utf-8") as f:
            return f.read()
    else:
        return carregar_prompt_padrao()

def salvar_prompt_custom(texto: str):
    with open(PROMPT_CUSTOM_FILE, "w", encoding="utf-8") as f:
        f.write(texto)

def restaurar_prompt_padrao():
    prompt_padrao = carregar_prompt_padrao()
    with open(PROMPT_CUSTOM_FILE, "w", encoding="utf-8") as f:
        f.write(prompt_padrao)

# Função para recarregar classificadores
def recarregar_classificadores(b):
    global regex_classifier, deepseek_classifier, gpt35_classifier, gemini_classifier

    with saida_recarregar:
        clear_output()

        deepseek_key = os.getenv("DEEPSEEK_API_KEY")
        gpt_key = os.getenv("OPENAI_API_KEY")
        gemini_key = os.getenv("GOOGLE_API_KEY")

        regex_classifier = RegexLegalClassifier()

        if deepseek_key:
            deepseek_classifier = DeepSeekLegalClassifier(deepseek_key)
        else:
            deepseek_classifier = None

        if gpt_key:
            gpt35_classifier = GPT35LegalClassifier(gpt_key)
        else:
            gpt35_classifier = None

        if gemini_key:
            gemini_classifier = GeminiLegalClassifier(gemini_key)
        else:
            gemini_classifier = None

        print("✅ Classificadores recarregados com sucesso usando o novo Prompt!")


# Widgets para edição de prompt
prompt_editor = widgets.Textarea(
    value=carregar_prompt_custom(),
    placeholder="Edite o prompt aqui...",
    description='Prompt:',
    layout=widgets.Layout(width='100%', height='300px')
)

botao_salvar_prompt = widgets.Button(
    description='💾 Salvar Alterações',
    button_style='success',
    layout=widgets.Layout(width='auto')
)

botao_restaurar_prompt = widgets.Button(
    description='♻️ Restaurar Padrão',
    button_style='warning',
    layout=widgets.Layout(width='auto')
)

botao_recarregar_classificadores = widgets.Button(
    description='🔄 Recarregar Classificadores',
    button_style='info',
    layout=widgets.Layout(width='auto')
)

saida_prompt = widgets.Output()

# Ações dos botões
def ao_clicar_salvar(b):
    with saida_prompt:
        clear_output()
        salvar_prompt_custom(prompt_editor.value)
        print("✅ Prompt salvo com sucesso! Agora clique em 🔄 Recarregar Classificadores.")

def ao_clicar_restaurar(b):
    with saida_prompt:
        clear_output()
        restaurar_prompt_padrao()
        prompt_editor.value = carregar_prompt_padrao()
        print("✅ Prompt restaurado para o padrão! Agora clique em 🔄 Recarregar Classificadores.")

def ao_clicar_recarregar(b):
    with saida_prompt:
        clear_output()
        recarregar_classificadores(b)
        print("✅ Classificadores recarregados com sucesso usando o novo Prompt!")

botao_salvar_prompt.on_click(ao_clicar_salvar)
botao_restaurar_prompt.on_click(ao_clicar_restaurar)
botao_recarregar_classificadores.on_click(ao_clicar_recarregar)

# Montagem visual
accordion_prompt = widgets.Accordion(children=[
    widgets.VBox([
        prompt_editor,
        widgets.HBox([botao_salvar_prompt, botao_restaurar_prompt, botao_recarregar_classificadores]),
        widgets.HTML("<br>"),
        saida_prompt
    ])
])
accordion_prompt.set_title(0, "⚙️ Edição e Atualização do Prompt")

display(accordion_prompt)


Accordion(children=(VBox(children=(Textarea(value='Você é um classificador jurídico especializado em documento…

In [ ]:
# === Botão para Recarregar Classificadores ===

import os
import ipywidgets as widgets
from IPython.display import display, clear_output
from modules.prompts.prompt_definitions import carregar_prompt_custom
from modules.classifiers.regex_classifier import RegexLegalClassifier
from modules.classifiers.deepseek_classifier import DeepSeekLegalClassifier
from modules.classifiers.gpt35_classifier import GPT35LegalClassifier
from modules.classifiers.gemini_classifier import GeminiLegalClassifier

# Botão
botao_recarregar_classificadores = widgets.Button(
    description='♻️ Recarregar Classificadores',
    button_style='info',
    layout=widgets.Layout(width='auto')
)

saida_recarregar = widgets.Output()

def recarregar_classificadores(b):
    global regex_classifier, deepseek_classifier, gpt35_classifier, gemini_classifier

    with saida_recarregar:
        clear_output()
        
        # Atualiza os classificadores carregando o novo prompt customizado
        deepseek_key = os.getenv("DEEPSEEK_API_KEY")
        gpt_key = os.getenv("OPENAI_API_KEY")
        gemini_key = os.getenv("GOOGLE_API_KEY")
        
        regex_classifier = RegexLegalClassifier()
        
        if deepseek_key:
            deepseek_classifier = DeepSeekLegalClassifier(deepseek_key)
        else:
            deepseek_classifier = None
        
        if gpt_key:
            gpt35_classifier = GPT35LegalClassifier(gpt_key)
        else:
            gpt35_classifier = None
        
        if gemini_key:
            gemini_classifier = GeminiLegalClassifier(gemini_key)
        else:
            gemini_classifier = None

        print("✅ Classificadores recarregados com o novo Prompt Customizado!")

botao_recarregar_classificadores.on_click(recarregar_classificadores)

# Exibe
display(widgets.VBox([
    widgets.HTML("<h3>🔄 Atualizar Classificadores Após Salvar Prompt</h3>"),
    botao_recarregar_classificadores,
    saida_recarregar
]))
